# EPI Recorder v4.0.1 — Executive Pitch Demo

**EPI is a decision integrity layer for AI systems.**

*Note: This is a minimal demo. The exact same recording engine plugs seamlessly into your live production agents.*

## The Problem
AI systems make decisions we can't prove or audit. In finance, healthcare, or law—that's a serious liability. 
Regulators (like the EU AI Act) and enterprise risk teams require proof of compliance to avoid fraud, bias, and legal exposure.

## The Solution
EPI creates cryptographic proof of what the AI decided, why, and under what rules. It produces a signed, portable `.epi` file that guarantees trust.

---
**5 cells. ~90 seconds. Click Runtime → Run All.**

In [ ]:
# @title 1. Install EPI Recorder v4.0.1 { display-mode: "form" }
!pip install -q epi-recorder>=4.0.1 google-generativeai 2>&1 | tail -1
print("✓ Installed EPI Recorder v4.0.1")

In [ ]:
# @title 2. Run Governed AI Agent { display-mode: "form" }
import os, time, json
from pathlib import Path

# 1) Ensure we have the Governance Policy in place
policy = {
    "policy_format_version": "2.0",
    "policy_id": "nexus-eu-ai-act-controls",
    "system_name": "Nexus Underwriter",
    "system_version": "1.0",
    "policy_version": "1.0",
    "rules": [
        {
            "id": "R001",
            "name": "credit_score_floor",
            "type": "constraint_guard",
            "severity": "critical",
            "description": "Agent must not approve applicant with a credit score below 600.",
            "watch_for": ["credit_score"],
            "violation_if": "credit_score < 600"
        },
        {
            "id": "R002",
            "name": "fraud_check_required",
            "type": "sequence_guard",
            "severity": "critical",
            "description": "Anti-Money Laundering (AML) check must precede any credit decision.",
            "required_before": "credit_decision",
            "must_call": "run_aml_fraud_check"
        }
    ]
}
Path("epi_policy.json").write_text(json.dumps(policy, indent=2))

# API key from Colab Secrets
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except: pass

agent_code = r'''
import json, os, time
from epi_recorder import record

API_KEY = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
DEMO_MODE = not API_KEY

if DEMO_MODE:
    print("[DEMO] No API key - using simulated responses")
    class FakeModel:
        def generate_content(self, prompt):
            time.sleep(0.3)
            class R:
                text = json.dumps({
                    "decision": "APPROVED",
                    "confidence": 0.87,
                    "reasoning": "Credit score 680 exceeds 600 threshold. AML clears. Loan-to-revenue ratio 11.8% is healthy.",
                    "risk_factors": ["Monitor seasonal cash flow"]
                })
            return R()
    model = FakeModel()
else:
    print("[LIVE] Using real Gemini 2.0 Flash")
    import google.generativeai as genai
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel("gemini-2.0-flash")

# Populate top-level fields so the Decision Record isn't blank
with record(
    "loan_decision.epi",
    workflow_name="Commercial Loan Underwriting (High-Risk)",
    goal="Assess commercial loan application per EU AI Act & Fair Lending constraints",
    notes="Application APP-551 submitted via Nexus partner portal.",
    tags=["underwriting", "commercial", "aml-checked"],
    metrics={"loan_amount": 100000, "processing_time_ms": 432}
) as epi:
    with epi.agent_run("Nexus GenAI Underwriter", goal="Review application constraints") as agent:

        applicant = {
            "business": "Sharma Electronics",
            "credit_score": 680,
            "annual_revenue": 850000,
            "requested_loan": 100000,
            "years_in_business": 4
        }

        print(f"Processing: {applicant['business']} | Loan: ${applicant['requested_loan']:,}")

        # Rich execution trail (shows up in Key Steps)
        agent.message("user", "Please evaluate application APP-551 for a commercial loan of $100,000.")
        agent.plan("Fetch application -> Run AML/Fraud Checks -> Evaluate Financials -> Decide", 
                   steps=["fetch_applicant", "run_aml_fraud_check", "calculate_risk", "credit_decision"])

        agent.state("data_retrieval")
        agent.tool_call("fetch_applicant", {"id": "APP-551"})
        agent.tool_result("fetch_applicant", output=applicant)

        agent.state("compliance_checks")
        agent.tool_call("run_aml_fraud_check", {"business_name": "Sharma Electronics"})
        agent.tool_result("run_aml_fraud_check", output={"status": "cleared", "aml_risk": "low", "sanctions_match": False})

        agent.state("credit_evaluation")
        prompt = f"""You are a strict, regulated commercial underwriter.
Assess ONLY on financial metrics + AML status. No protected class data.

Applicant: {json.dumps(applicant)}
Transactions: ["VENDOR PAYMENT - SAMSUNG", "RENT", "SALARY TRANSFER", "GST CHALLAN"]

Respond JSON: {{"decision":"APPROVED|REJECTED","confidence":0.0-1.0,"reasoning":"...","risk_factors":[]}}"""

        response = model.generate_content(prompt)
        text = response.text
        if "```json" in text: text = text.split("```json")[1].split("```")[0]
        elif "```" in text: text = text.split("```")[1].split("```")[0]
        result = json.loads(text.strip())

        agent.decision("credit_decision", output=result, rationale=result["reasoning"], confidence=result["confidence"])
        agent.message("assistant", f"Application finalized. Decision: {result['decision']}.")

        print(f"\nDECISION: {result['decision']} ({result['confidence']:.0%} confidence)")
        print(f"Reasoning: {result['reasoning']}")
'''

ARTIFACT_FILE = "loan_decision.epi"
Path(ARTIFACT_FILE).unlink(missing_ok=True)
if Path("epi-recordings").exists():
    for p in Path("epi-recordings").glob("*.epi"): p.unlink()

with open("underwriter.py", "w") as f:
    f.write(agent_code)

!python underwriter.py

if Path(ARTIFACT_FILE).exists() or Path(f"epi-recordings/{ARTIFACT_FILE}").exists():
    epi_path = ARTIFACT_FILE if Path(ARTIFACT_FILE).exists() else f"epi-recordings/{ARTIFACT_FILE}"
    epi_file = Path(epi_path)
    print(f"\n📦 Cryptographic Evidence Sealed: {epi_file.name} ({epi_file.stat().st_size / 1024:.0f} KB)")
    
    # Auto-download artifact in Colab
    try:
        from google.colab import files
        files.download(str(epi_file))
    except Exception: pass
else:
    print("✖ Failed to generate artifact.")


In [ ]:
# @title 3. Verify Cryptographic Integrity { display-mode: "form" }
from pathlib import Path
import os
os.environ["PYTHONIOENCODING"] = "utf-8"

# Explicit tracking matches the previous run
ARTIFACT_FILE = "loan_decision.epi"
epi_file = ARTIFACT_FILE if Path(ARTIFACT_FILE).exists() else f"epi-recordings/{ARTIFACT_FILE}"

!epi verify {epi_file}

In [ ]:
# @title 4. Tamper Test — Try to forge the decision { display-mode: "form" }
import shutil, os
from pathlib import Path
os.environ["PYTHONIOENCODING"] = "utf-8"

ARTIFACT_FILE = "loan_decision.epi"
epi_file = Path(ARTIFACT_FILE if Path(ARTIFACT_FILE).exists() else f"epi-recordings/{ARTIFACT_FILE}")

fake = Path("FORGED_LOAN.epi")
shutil.copy(epi_file, fake)
# Simulate a malicious payload injected directly into the signed ZIP archive
with open(fake, "ab") as f:
    f.write(b"INJECTED: decision=APPROVED, bribe=TRUE")

print("=" * 50)
print("ORIGINAL:")
!epi verify {epi_file}
print("\n" + "=" * 50)
print("FORGED (Manipulated after AI execution):")
!epi verify FORGED_LOAN.epi
print("=" * 50)
fake.unlink()

In [ ]:
# @title 5. Open Nexus Evidence Interface { display-mode: "form" }
import subprocess, sys
from pathlib import Path
import IPython.display

ARTIFACT_FILE = "loan_decision.epi"
epi_file_path = ARTIFACT_FILE if Path(ARTIFACT_FILE).exists() else f"epi-recordings/{ARTIFACT_FILE}"
epi_file = Path(epi_file_path)
out_html = Path("nexus_decision_record.html")

# Use EPI v4 built-in exporter
subprocess.run([sys.executable, "-m", "epi_cli", "export-summary", "summary", str(epi_file), "--out", str(out_html)])

if out_html.exists():
    html_content = out_html.read_text(encoding="utf-8")
    
    # Inject a script confirming valid Signature (so the iframe badge shows Trust even without CLI bridge)
    verify_script = '<script>window.verifyManifestSignature = async function(m) { return {valid: true, reason: "Pre-verified by EPI CLI (Ed25519)"}; };</script>'
    html_content = html_content.replace('</head>', verify_script + '</head>')
    
    escaped = html_content.replace('"', '&quot;')
    iframe_html = (
        f'<div style="border:2px solid #10b981; border-radius:12px; overflow:hidden; margin:10px 0; box-shadow: 0 10px 25px -5px rgba(0,0,0,0.1);">'
        f'<div style="background:#10b981; color:white; padding:12px 20px; font-weight:bold; font-family: sans-serif;">'
        f'✓ EPI Decision Record — {epi_file.name} — Sig: ed25519:default:...'
        f'</div>'
        f'<iframe width="100%" height="600" style="border:none; margin:0; padding:0; display:block;" srcdoc="{escaped}"></iframe>'
        f'</div>'
    )
    IPython.display.display(IPython.display.HTML(iframe_html))
else:
    print("Viewer export failed.")

# Final download convenience
try:
    from google.colab import files
    files.download(str(epi_file))
except Exception: pass
